In [0]:
%sql
USE CATALOG workspace;
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;
SHOW SCHEMAS;


In [0]:
%sql
COMMENT ON SCHEMA bronze IS 'Raw ingested events with minimal transformation';
COMMENT ON SCHEMA silver IS 'Validated, deduped, enriched events';
COMMENT ON SCHEMA gold IS 'Business-ready aggregates and curated views';

In [0]:
BRONZE_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events"
SILVER_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events"

In [0]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

(bronze_df.write
  .mode("overwrite")
  .saveAsTable("bronze.events"))


In [0]:
# Register Silver if it exists
try:
    silver_df = spark.read.format("delta").load(SILVER_PATH)
    (silver_df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable("silver.events"))
except Exception as e:
    print("Silver not registered yet:", e)


In [0]:
%sql
COMMENT ON TABLE bronze.events IS 'Bronze events registered from Delta files stored in Volumes';
COMMENT ON VIEW gold.top_products IS 'Top products by purchase count (controlled access view)';
ALTER TABLE bronze.events ALTER COLUMN user_id COMMENT 'User identifier (sensitive)';


In [0]:
%sql
-- Business view (Top products by purchases)
CREATE OR REPLACE VIEW gold.top_products AS
SELECT
  product_id,
  COUNT(*) AS purchases
FROM bronze.events
WHERE event_type = 'purchase'
GROUP BY product_id
HAVING COUNT(*) > 10
ORDER BY purchases DESC
LIMIT 100;


In [0]:
%sql
SELECT COUNT(*) AS bronze_rows FROM bronze.events;
-- SELECT * FROM gold.top_products
-- limit 10;

In [0]:
%sql
GRANT SELECT ON VIEW gold.top_products TO `neo@gmail.com`;

In [0]:
%sql
-- Controlled access views
CREATE OR REPLACE VIEW gold.v_events_analytics AS
SELECT
  event_type,
  product_id,
  category_code,
  brand,
  price
FROM bronze.events;

select * from gold.v_events_analytics
limit 10;
